In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "AVAXUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-09-01 00:00:00+00:00,23.40,23.40,23.34,23.36,1949.78,2025-09-01 00:00:59.999999+00:00,45551.6510,245,990.53,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000000,0.000000e+00,0.000000,NaN,NaN
1,2025-09-01 00:01:00+00:00,23.37,23.38,23.36,23.38,2749.32,2025-09-01 00:01:59.999999+00:00,64252.5574,117,1277.50,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000449,2.492877e-04,0.000199,NaN,NaN
2,2025-09-01 00:02:00+00:00,23.37,23.37,23.34,23.35,2469.23,2025-09-01 00:02:59.999999+00:00,57645.9984,202,540.23,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.000359,1.770022e-07,-0.000359,NaN,NaN
3,2025-09-01 00:03:00+00:00,23.35,23.36,23.33,23.34,1112.24,2025-09-01 00:03:59.999999+00:00,25958.7190,136,289.20,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.001078,-3.650454e-04,-0.000713,NaN,NaN
4,2025-09-01 00:04:00+00:00,23.34,23.34,23.27,23.28,15199.03,2025-09-01 00:04:59.999999+00:00,354054.1022,540,5331.10,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.003834,-1.396899e-03,-0.002437,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 263,876
[info] optuna train rows: 168,880
[info] valid rows:        42,220
[info] test rows:         52,776


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-20 04:28:09,218] A new study created in memory with name: no-name-17fedf03-201b-4f1a-aad9-7e9ee6eff754


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:09<?, ?it/s]

Best trial: 0. Best value: 0.071204:   0%|          | 0/50 [00:09<?, ?it/s]

Best trial: 0. Best value: 0.071204:   2%|▏         | 1/50 [00:09<07:57,  9.75s/it]

...

Best trial: 33. Best value: 0.0798095:  98%|█████████▊| 49/50 [30:37<00:19, 19.73s/it]

Best trial: 33. Best value: 0.0798095:  98%|█████████▊| 49/50 [30:37<00:19, 19.73s/it]

Best trial: 33. Best value: 0.0798095: 100%|██████████| 50/50 [30:37<00:00, 34.86s/it]

Best trial: 33. Best value: 0.0798095: 100%|██████████| 50/50 [30:37<00:00, 36.74s/it]

[I 2026-03-20 04:58:46,398] Trial 49 finished with value: 0.06677737229153337 and parameters: {'n_estimators': 800, 'max_depth': 17, 'min_samples_split': 21, 'min_samples_leaf': 4, 'max_features': 0.8, 'bootstrap': True}. Best is trial 33 with value: 0.07980952579097855.

[optuna] best trial
value: 0.079810
params:
  n_estimators: 800
  max_depth: 19
  min_samples_split: 25
  min_samples_leaf: 5
  max_features: log2
  bootstrap: False


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final rf...


[training] done in 15.42s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.729791
Test IC:       -0.042255
Train Rank IC: 0.156955
Test Rank IC:  0.062069
Train RMSE:    0.003783
Test RMSE:     0.002832


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
mom_3               0.135946
mom_30              0.099940
mom_15              0.093790
mom_5               0.082163
dist_ma_15          0.075095
mom_60              0.074803
dist_ma_30          0.070176
dist_ma_5           0.059122
mom_x_imb           0.052125
mom_10              0.049952
macd_hist           0.033663
bar_range           0.032668
range_5             0.022045
vol_15              0.018973
vol_5               0.012951
range_15            0.012578
range_ratio         0.012033
atr_norm            0.011818
vol_30              0.011105
mr_x_vol            0.005390
dist_ma_15_z        0.004640
trend_x_imb         0.004566
imbalance_15        0.003890
vol_regime_ratio    0.003167
trend_strength      0.002984
imbalance_5         0.002863
num_trades_mom_5    0.002293
volume_mom_5        0.001793
imbalance           0.000980
vol_ratio_5_30      0.000857
dom_sin             0.000731
trades_z            0.000718
hour_cos            0.000673
is_high_vol

In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/rf/AVAXUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/rf/AVAXUSDT__h5_model.joblib
[saved] features -> models/rf/AVAXUSDT__h5_feature_cols.json
[saved] feature importance -> models/rf/AVAXUSDT__h5_feature_importance.csv
[saved] metadata -> models/rf/AVAXUSDT__h5_meta.json
